In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import cvxpy as cp
from sklearn.covariance import LedoitWolf

In [ ]:
class FactorModel:
  def __init__(self, rfr, **kwargs):
    self.rfr = rfr
    super().__init__(**kwargs)

  def get_factors_exp(self, returns, factors, verbose=False):
    y = returns.sub(self.rfr, axis=0) if hasattr(self.rfr, "index") else returns - self.rfr
    X = sm.add_constant(factors)

    rows = {}
    for t in y.columns:
      model = sm.OLS(y[t], X, missing="drop").fit()
      row = {"alpha": model.params["const"]}
      row.update({f"beta_{f}": model.params[f] for f in factors.columns})
      rows[t] = row

      if verbose:
        print(f"-------------- {t:^6} --------------")
        print(model.params.to_string(float_format=lambda v: f"{v:.6f}"))
        print(f"R-Squared (R2): {model.rsquared:.4f}\n")

    return pd.DataFrame.from_dict(rows, orient="index")


In [ ]:
class Optimizer:
  def __init__(self, risk_aversion=2, max_exp=1.5, max_w=0.15, **kwargs):
    self.risk_aversion = float(risk_aversion)
    self.max_exp = float(max_exp)
    self.max_w = max_w
    super().__init__(**kwargs)

  def _optimize_w(self, alphas, cov, betas, signs, borrow_rates=None):
    alphas = np.asarray(alphas, dtype=float)
    cov = np.asarray(cov, dtype=float)
    betas = np.asarray(betas, dtype=float)
    signs = np.asarray(signs, dtype=float)

    w = cp.Variable(len(alphas))

    exp_alphas = alphas @ w
    risk_term = 0.5 * self.risk_aversion * cp.quad_form(w, cp.psd_wrap(cov))

    borrow_penalty = (
        np.asarray(borrow_rates, dtype=float) @ cp.neg(w)
        if borrow_rates is not None else 0
    )

    constraints = [
        cp.norm(w, 1) <= self.max_exp,
        betas @ w == 0,
        cp.multiply(signs, w) >= 0,
        cp.abs(w) <= self.max_w,
    ]

    prob = cp.Problem(cp.Maximize(exp_alphas - risk_term - borrow_penalty), constraints)
    prob.solve(solver=cp.CLARABEL)

    if prob.status != cp.OPTIMAL:
      raise ValueError(f"Optimization failed: {prob.status}")

    return w.value

In [ ]:
class Portfolio(FactorModel, Optimizer):
  def __init__(self, universe, **kwargs):
    self.universe = list(universe.keys())
    self.signs = list(universe.values())
    super().__init__(**kwargs)

  def get_data(self, start="2020-01-01", end="2026-10-01", benchmark="^GSPC"):
    prices = yf.download(self.universe + [benchmark], start=start, end=end)["Close"]
    rets = prices.pct_change(fill_method=None).dropna()

    return rets[self.universe], rets[[benchmark]]

  def _get_factor_exposures(self, returns, benchmark):
    factors = (benchmark.sub(self.rfr, axis=0)
               if hasattr(self.rfr, "index") else benchmark - self.rfr)
    exposures = self.get_factors_exp(returns, factors)
    alphas = exposures["alpha"]
    B = exposures.filter(like="beta_")
    return alphas, B

  def get_portfolio(self, returns, benchmark):
    alphas, B = self._get_factor_exposures(returns, benchmark)
    cov = LedoitWolf().fit(returns).covariance_
    w = self._optimize_w(
        alphas=alphas, cov=cov, betas=B.to_numpy().T,
        signs=self.signs,
    )
    return pd.Series(w, index=returns.columns, name="weight")

In [ ]:
def rolling_backtest(ptf, returns, benchmark, lookback=504, rebalance=21,
                     cost_bps=5, periods=252):
    cost = cost_bps / 1e4
    w_cur = pd.Series(0.0, index=returns.columns)
    port, turnover, weights = {}, {}, {}

    for start in range(lookback, len(returns), rebalance):
        train_r = returns.iloc[start - lookback:start]
        train_b = benchmark.iloc[start - lookback:start]
        hold = returns.iloc[start:start + rebalance]

        try:
            w_new = ptf.get_portfolio(train_r, train_b)
        except ValueError as e:
            print(f"{returns.index[start].date()}: {e}; keeping previous weights")
            w_new = w_cur

        turnover[returns.index[start]] = (w_new - w_cur).abs().sum()
        weights[returns.index[start]] = w_new
        w = w_new.copy()

        for i, (d, r) in enumerate(hold.iterrows()):
            pr = float((w * r).sum())
            if i == 0:
                pr -= cost * turnover[returns.index[start]]
            port[d] = pr
            w = w * (1 + r) / (1 + pr)
        w_cur = w

    port = pd.Series(port, name="strategy")
    bench = benchmark.iloc[:, 0].reindex(port.index)

    X = np.column_stack([np.ones(len(port)), bench.values])
    alpha_d, beta = np.linalg.lstsq(X, port.values, rcond=None)[0]
    equity = (1 + port).cumprod()

    stats = pd.Series({
        "ann_return": (1 + port).prod() ** (periods / len(port)) - 1,
        "ann_vol": port.std() * np.sqrt(periods),
        "sharpe": port.mean() / port.std() * np.sqrt(periods),
        "max_drawdown": (equity / equity.cummax() - 1).min(),
        "realized_beta": beta,
        "ann_alpha": alpha_d * periods,
        "avg_turnover_per_rebal": np.mean(list(turnover.values())),
        "bench_ann_return": (1 + bench).prod() ** (periods / len(bench)) - 1,
    })

    return {
        "returns": port,
        "equity": equity,
        "weights": pd.DataFrame(weights).T,
        "turnover": pd.Series(turnover),
        "stats": stats,
    }


def plot_backtest(res, benchmark):
    eq = res["equity"]
    bench_eq = (1 + benchmark.iloc[:, 0].reindex(eq.index)).cumprod()
    fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
    eq.plot(ax=ax[0], label="long/short")
    bench_eq.plot(ax=ax[0], label="benchmark", alpha=0.7)
    ax[0].legend(); ax[0].set_title("Equity curve")
    (eq / eq.cummax() - 1).plot(ax=ax[1], color="firebrick")
    ax[1].set_title("Drawdown")
    plt.tight_layout(); plt.show()